# RSNA Knee — Model Inspection & Case Review
Interactive companion while training runs. Works with whatever folds have finished — re-run cells any time.

1) live curves · 2) per-label AUC (best checkpoints) · 3) gold-58 case browser · 4) model-vs-teacher disagreement browser · 5) weak-label review queue (your expert eyes on Synovitis / PF OA / Contusion)

In [ ]:
from pathlib import Path
import json, re, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

random.seed()  # fresh sample each re-run
WORK = Path("/work/07880/devansh/lonestar/rsna_knee")
DATA = Path("/scratch/07880/devansh/rsna_knee/data")
PROC = DATA/"proc/train"
LABELS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
          "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
train  = pd.read_csv(DATA/"csv/train.csv").set_index("StudyInstanceUID")
series = pd.read_csv(DATA/"csv/train_series.csv")
folds  = pd.read_csv(DATA/"csv/train_folds.csv").set_index("StudyInstanceUID")
IMGV = WORK/"results/img_v1"

def show_study(uid, title_extra=""):
    rows = series[series.StudyInstanceUID == uid]
    fig, axes = plt.subplots(1, len(rows), figsize=(3.0*len(rows), 3.2))
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, rows.itertuples()):
        p = PROC/uid/f"{r.SeriesInstanceUID}.npy"
        ax.axis("off")
        if p.exists():
            v = np.load(p, mmap_mode="r")
            ax.imshow(v[v.shape[0]//2], cmap="gray")
            ax.set_title(f"{r.Anatomical_Plane[:3]} f{r.Fluid_Sensitive}", fontsize=8)
    plt.suptitle(f"{uid[-12:]} {title_extra}", fontsize=9); plt.tight_layout(); plt.show()

def show_series_sweep(uid, sid=None, n=12):
    rows = series[series.StudyInstanceUID == uid]
    if sid is None:  # prefer fluid-sensitive sagittal
        pref = rows[(rows.Anatomical_Plane=="Sagittal") & (rows.Fluid_Sensitive==1)]
        sid = (pref.iloc[0] if len(pref) else rows.iloc[0]).SeriesInstanceUID
    v = np.load(PROC/uid/f"{sid}.npy")
    idx = np.linspace(0, v.shape[0]-1, n).astype(int)
    fig, axes = plt.subplots(2, n//2, figsize=(16, 5.5))
    for ax, i in zip(axes.ravel(), idx):
        ax.imshow(v[i], cmap="gray"); ax.set_title(f"{i}", fontsize=7); ax.axis("off")
    plt.tight_layout(); plt.show()
print("helpers ready")

## 1. Live training curves

In [ ]:
hist = {}
for f in range(4):
    p = WORK/f"logs/train_fold{f}.log"
    if p.exists():
        rows = re.findall(r"ep(\d+): loss ([\d.]+) \| val-teacher AUC ([\d.]+) \| gold AUC ([\d.]+)", p.read_text())
        if rows: hist[f] = pd.DataFrame(rows, columns=["ep","loss","val","gold"]).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for f, h in hist.items():
    axes[0].plot(h.ep, h.loss, marker="o", label=f"f{f}")
    axes[1].plot(h.ep, h.val, marker="o", label=f"f{f}")
    axes[2].plot(h.ep, h.gold, marker="o", label=f"f{f}")
for ax, t in zip(axes, ["train loss","val AUC vs teacher","AUC vs gold-58"]):
    ax.set_title(t); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 2. Per-label AUC from finished folds (out-of-fold vs teacher, and vs gold)

In [ ]:
from sklearn.metrics import roc_auc_score
oof, gold_preds = [], []
for f in range(4):
    vp = IMGV/f"fold{f}/val_preds.csv"
    gp = IMGV/f"fold{f}/gold_preds.csv"
    if vp.exists(): oof.append(pd.read_csv(vp))
    if gp.exists(): gold_preds.append(pd.read_csv(gp).set_index("StudyInstanceUID")[LABELS])
print(f"finished folds: {len(oof)}")
if oof:
    o = pd.concat(oof).set_index("StudyInstanceUID")
    rows = []
    for lab in LABELS:
        th = (folds.loc[o.index, f"t_{lab}"] >= 0.5).astype(int)
        auc_t = roc_auc_score(th, o[lab]) if 0 < th.mean() < 1 else np.nan
        auc_g = np.nan
        if gold_preds:
            gp_m = sum(gold_preds)/len(gold_preds)  # avg fold models on gold
            gt = train.loc[gp_m.index, lab].astype(int)
            auc_g = roc_auc_score(gt, gp_m[lab])
        rows.append(dict(label=lab, oof_auc_vs_teacher=auc_t, gold_auc_ensembled=auc_g))
    r = pd.DataFrame(rows).set_index("label")
    r.loc["mean"] = r.mean()
    display(r.round(3))

## 3. Gold-58 case browser
Ground truth + teacher + model side by side. Re-run for a new case, or set `uid` manually.

In [ ]:
gold_uids = folds[folds.is_gold == 1].index.tolist()
uid = random.choice(gold_uids)
gt = train.loc[uid, LABELS].astype(float)
tt = folds.loc[uid, [f"t_{l}" for l in LABELS]].values.astype(float)
comp = pd.DataFrame({"gold": gt.values, "teacher": tt.round(2)}, index=LABELS)
for f in range(4):
    gp = IMGV/f"fold{f}/gold_preds.csv"
    if gp.exists():
        g = pd.read_csv(gp).set_index("StudyInstanceUID")
        if uid in g.index: comp[f"model_f{f}"] = g.loc[uid, LABELS].round(2).values
display(comp.T)
show_study(uid); show_series_sweep(uid)
print(train.loc[uid, "Report"][:1500])

## 4. Model vs teacher — biggest disagreements (finished folds)
Large |model − teacher| on a confident teacher score = either a model failure **or a report-invisible finding the image model caught**. Both are informative.

In [ ]:
LAB = "Effusion"   # <- change label here
if oof:
    o = pd.concat(oof).set_index("StudyInstanceUID")
    t = folds.loc[o.index, f"t_{LAB}"]
    d = (o[LAB] - t).abs().sort_values(ascending=False)
    top = d.head(8)
    for uid, gap in top.items():
        print(f"model={o.loc[uid, LAB]:.2f} teacher={t.loc[uid]:.2f} gap={gap:.2f}")
        show_study(uid, f"{LAB}: model {o.loc[uid, LAB]:.2f} vs teacher {t.loc[uid]:.2f}")
        print(train.loc[uid, "Report"][:400], "\n" + "-"*80)

## 5. Weak-label review queue (expert annotation)
Teacher is weakest on **Synovitis (AUC .73), PF OA (.79), Contusion (.80)**. Reviewing borderline cases yourself (images + report) and recording your call creates a small high-quality label set we can use to recalibrate the teacher or fine-tune the model head.
Your calls are appended to `results/expert_review.csv` — aim for a batch of 20-30 per sitting.

In [ ]:
REVIEW_LAB = "Synovitis"    # <- change label
band = folds[(folds[f"t_{REVIEW_LAB}"] > 0.25) & (folds[f"t_{REVIEW_LAB}"] < 0.75) & (folds.is_gold == 0)]
uid = random.choice(band.index.tolist())
print(f"{REVIEW_LAB}: teacher={folds.loc[uid, f't_{REVIEW_LAB}']:.2f}   study ...{uid[-12:]}")
show_study(uid); show_series_sweep(uid)
print(train.loc[uid, "Report"][:1500])

In [ ]:
# record your call for the study above, then re-run the cell before this one for the next case
MY_CALL = None     # set to 1 (present) or 0 (absent), then run
if MY_CALL is not None:
    out = WORK/"results/expert_review.csv"
    row = pd.DataFrame([dict(StudyInstanceUID=uid, label=REVIEW_LAB, call=int(MY_CALL),
                             teacher=float(folds.loc[uid, f"t_{REVIEW_LAB}"]))])
    row.to_csv(out, mode="a", header=not out.exists(), index=False)
    print(f"saved -> {out} ({sum(1 for _ in open(out))-1} total calls)")